In [23]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
from qiskit_aer import AerSimulator
from qiskit.circuit import Parameter
import numpy as np

In [21]:
alpha = Parameter('alpha')
beta = Parameter('beta')


In [7]:
backend = AerSimulator()

In [64]:
qc = QuantumCircuit(6,4)
qc.unitary(
    np.array([
        [3**0.5 /2, -1/2],
        [1/2, 3**0.5 /2]
    ]), 
    [0]
)
qc.unitary(
    np.array([
        [(2/3)**0.5, -(1/3)**0.5],
        [(1/3)**0.5, (2/3)**0.5]
    ]), 
    [5]
)
# qc.rx(np.pi/8, 5)


qc.h(1)
qc.h(2)
qc.cx(1,3)
qc.cx(2,4)
qc.barrier()
G_HH = 2**-0.5 * np.array([
    [1,0,0,1],
    [0,1,1,0],
    [0,1,-1,0],
    [1,0,0,-1]
])
qc.unitary(G_HH, (0, 1))
qc.unitary(G_HH, (4, 5))
qc.barrier()
qc.save_statevector('pre-measure')
qc.measure(0,0)
qc.measure(1,1)
qc.measure(4,2)
qc.measure(5,3)
qc.save_statevector()
res = backend.run(qc).result()
sv = res.data()['statevector']
sv

Statevector([ 0.        +0.j,  0.        +0.j,  0.        +0.j,
              0.        +0.j,  0.        +0.j,  0.        +0.j,
              0.        +0.j,  0.        +0.j,  0.        +0.j,
              0.        +0.j, -0.        +0.j, -0.        +0.j,
              0.        +0.j,  0.        +0.j, -0.        +0.j,
             -0.        +0.j,  0.5       +0.j,  0.        +0.j,
              0.        +0.j,  0.        +0.j,  0.70710678+0.j,
              0.        +0.j,  0.        +0.j,  0.        +0.j,
              0.28867513+0.j,  0.        +0.j, -0.        +0.j,
             -0.        +0.j,  0.40824829+0.j,  0.        +0.j,
             -0.        +0.j, -0.        +0.j, -0.        +0.j,
             -0.        +0.j, -0.        +0.j, -0.        +0.j,
              0.        +0.j,  0.        +0.j,  0.        +0.j,
              0.        +0.j, -0.        +0.j, -0.        +0.j,
              0.        +0.j,  0.        +0.j,  0.        +0.j,
              0.        +0.j, -0.       

In [66]:
from itertools import product, repeat

In [84]:
np.array([(2/3)**0.5 * (3/4)**0.5, (1/3)**0.5 * (3/4)**0.5, (2/3)**0.5 * (1/4)**0.5, (1/3)**0.5 * (1/4)**0.5])

array([0.70710678, 0.5       , 0.40824829, 0.28867513])

In [90]:
data = np.array(res.data()['pre-measure'],dtype=float)
data = data.reshape((2,2,2,2,2,2))
for a,b,c,d in product([0,1],repeat=4):
    print(a,b,c,d)
    red_sv = data[a,b,:,:,c,d].reshape((4,))
    out = (np.real(red_sv / (red_sv @ red_sv)**0.5))
    print(out)
    ops = [
        [np.array([[1,0],[0,1]]),                            np.array([[0,1],[1,0]])],
        [np.array([[1,0],[0,-1]]) @ np.array([[0,1],[1,0]]), np.array([[1,0],[0,-1]])]
    ]
    print(np.kron(ops[c][d], ops[a][b]) @ out )
    print('------------------')

0 0 0 0
[0.70710678 0.5        0.40824829 0.28867513]
[0.70710678 0.5        0.40824829 0.28867513]
------------------
0 0 0 1
[0.40824829 0.28867513 0.70710678 0.5       ]
[0.70710678 0.5        0.40824829 0.28867513]
------------------
0 0 1 0
[ 0.40824829  0.28867513 -0.70710678 -0.5       ]
[-0.70710678 -0.5        -0.40824829 -0.28867513]
------------------
0 0 1 1
[ 0.70710678  0.5        -0.40824829 -0.28867513]
[0.70710678 0.5        0.40824829 0.28867513]
------------------
0 1 0 0
[0.5        0.70710678 0.28867513 0.40824829]
[0.70710678 0.5        0.40824829 0.28867513]
------------------
0 1 0 1
[0.28867513 0.40824829 0.5        0.70710678]
[0.70710678 0.5        0.40824829 0.28867513]
------------------
0 1 1 0
[ 0.28867513  0.40824829 -0.5        -0.70710678]
[-0.70710678 -0.5        -0.40824829 -0.28867513]
------------------
0 1 1 1
[ 0.5         0.70710678 -0.28867513 -0.40824829]
[0.70710678 0.5        0.40824829 0.28867513]
------------------
1 0 0 0
[-0.5         0.

/nfs/users/nfs_j/jc59/quantumwork/pangenome/.venv/lib/python3.10/site-packages/qiskit/quantum_info/states/statevector.py:118: ComplexWarning: Casting complex values to real discards the imaginary part
  return np.array(self.data, dtype=dtype, copy=copy)


In [62]:
arr = np.array(sv)
np.real(arr[np.nonzero(arr)])

array([ 0.28867513,  0.40824829, -0.5       , -0.70710678])

In [63]:
np.array([(2/3)**0.5 * (3/4)**0.5, (2/3)**0.5 * (1/4)**0.5, (1/3)**0.5 * (3/4)**0.5, (1/3)**0.5 * (1/4)**0.5])

array([0.70710678, 0.40824829, 0.5       , 0.28867513])

In [55]:
sv = Statevector(sv)
# sv.measure([0,1,4,5])
sv.draw('latex')

<IPython.core.display.Latex object>

In [ ]:
goal = Statevector([(2/3)**0.5 * (3/4)**0.5,(2/3)**0.5 * (3/4)**0.5,(2/3)**0.5 * (3/4)**0.5,(2/3)**0.5 * (3/4)**0.5])
goal.draw('latex')

<IPython.core.display.Latex object>

In [50]:
qc.draw(fold=-1)


┌─────────┐           ░ ┌──────────┐ ░ ┌─┐          statevector 
q_0: ┤ Unitary ├───────────░─┤0         ├─░─┤M├───────────────░──────
     ├─────────┤┌───┐      ░ │  Unitary │ ░ └╥┘┌─┐            ░      
q_1: ┤ Unitary ├┤ H ├──■───░─┤1         ├─░──╫─┤M├────────────░──────
     └──┬───┬──┘└───┘  │   ░ └──────────┘ ░  ║ └╥┘            ░      
q_2: ───┤ H ├─────■────┼───░──────────────░──╫──╫─────────────░──────
        └───┘     │  ┌─┴─┐ ░              ░  ║  ║             ░      
q_3: ─────────────┼──┤ X ├─░──────────────░──╫──╫─────────────░──────
                ┌─┴─┐└───┘ ░ ┌──────────┐ ░  ║  ║ ┌─┐         ░      
q_4: ───────────┤ X ├──────░─┤0         ├─░──╫──╫─┤M├─────────░──────
                └───┘      ░ │  Unitary │ ░  ║  ║ └╥┘┌─┐      ░      
q_5: ──────────────────────░─┤1         ├─░──╫──╫──╫─┤M├──────░──────
                           ░ └──────────┘ ░  ║  ║  ║ └╥┘      ░      
c: 4/════════════════════════════════════════╩══╩══╩══╩══════════════
                                             0  1  2  3

In [36]:
G_HH @ G_HH.conj()

array([[0.5, 0. , 0. , 0. ],
       [0. , 0.5, 0. , 0. ],
       [0. , 0. , 0.5, 0. ],
       [0. , 0. , 0. , 0.5]])